# Мерей: воспроизведение runtime 29419f43

Этот notebook закрепляет runtime commit, запускает тесты и обязательную
проверку допуска/качества. Его подготовка не является выполненным Kaggle run.
Основное performance сравнение — **текущий Мерей против нового runtime**;
optional A/B находится в конце. Official harness также запускает native Soup,
поскольку это требуется для его admission/quality checks.

Kaggle: **GPU T4 x2**, Internet On, свободные GPU, свежая сессия, Run All.
Все GPU subprocesses выполняются последовательно. Компиляция учитывается
исходным harness; готовые compiler caches не поставляются. Сохраните version
с output и выдаваемую папку evidence. Final platform submission оформляется отдельно.


In [ ]:
import base64, hashlib, importlib.metadata as metadata, json, os
import shutil, statistics, subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/promaladoi/Decentrathon-6.0.git'
CANDIDATE = '29419f43d9b9173d1ba5d6824a31589ad3b84df7'
CURRENT_MEREI = 'd6dca3ac371b6bf5ac15584cd9afbd027aab4a0b'
RUN_CURRENT_MEREI_AB = False  # True adds old,new,new,old diagnostic runs.
MEMORY_CEILING_MIB = 5275
WORKING = Path('/kaggle/working')
EVIDENCE = WORKING / ('merey-reproduction-' + CANDIDATE[:12] + '-evidence')
EVIDENCE.mkdir(exist_ok=False)

gpu_names = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True)
print(gpu_names, flush=True)
assert 'T4' in gpu_names, 'Choose GPU T4 x2'
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    pass


In [ ]:
def installed(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

# Keep the recorded GPU image; do not download or replace PyTorch/Triton.
assert installed('torch') == '2.11.0+cu128', 'Use the recorded Kaggle image'
assert installed('triton') == '3.6.0', 'Use Triton 3.6.0 from the recorded image'
requirements = {
    'soup-cli': ('0.77.0', 'soup-cli[train]==0.77.0'),
    'transformers': ('5.16.1', 'transformers==5.16.1'),
    'peft': ('0.20.0', 'peft==0.20.0'),
    'trl': ('0.29.1', 'trl==0.29.1'),
    'bitsandbytes': ('0.50.2', 'bitsandbytes==0.50.2'),
    'accelerate': ('1.14.0', 'accelerate==1.14.0'),
    'datasets': ('4.8.5', 'datasets==4.8.5'),
}
missing = [spec for name, (required, spec) in requirements.items()
           if installed(name) != required]
if installed('pytest') is None:
    missing.append('pytest')
if installed('torchao') is not None:
    # Optional incompatible package is removed before all tests/runs.
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
versions = {name: installed(name) for name in ['torch', 'triton', *requirements, 'pytest']}
assert all(versions[name] == required for name, (required, spec) in requirements.items())
assert versions['torch'] == '2.11.0+cu128' and versions['triton'] == '3.6.0'
print(json.dumps(versions, indent=2), flush=True)


In [ ]:
token = ''
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
except Exception:
    pass

def clone_exact(label, commit):
    checkout = WORKING / ('merey-' + label + '-' + commit[:12])
    assert not checkout.exists(), 'Preserve previous results and use a fresh session'
    auth = []
    if token:
        basic = base64.b64encode(('x-access-token:' + token).encode()).decode()
        auth = ['-c', 'http.extraHeader=Authorization: Basic ' + basic]
    result = subprocess.run(['git', *auth, 'clone', '-q', '--no-checkout',
                             REPO_URL, str(checkout)], capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError('Git clone failed; check repository access/Kaggle Secrets')
    subprocess.run(['git', '-C', str(checkout), 'checkout', '-q', commit], check=True)
    actual = subprocess.check_output(
        ['git', '-C', str(checkout), 'rev-parse', 'HEAD'], text=True).strip()
    assert actual == commit
    return checkout / 'Мерей'

roots = {'candidate': clone_exact('runtime', CANDIDATE)}
if RUN_CURRENT_MEREI_AB:
    roots['current'] = clone_exact('current', CURRENT_MEREI)
del token
if RUN_CURRENT_MEREI_AB:
    for directory in ['harness', 'task', 'tests']:
        a, b = roots['current'] / directory, roots['candidate'] / directory
        files = sorted(p.relative_to(a) for p in a.rglob('*') if p.is_file())
        assert files == sorted(p.relative_to(b) for p in b.rglob('*') if p.is_file())
        assert all((a / p).read_bytes() == (b / p).read_bytes() for p in files), directory

sys.path.insert(0, str(roots['candidate'] / 'harness'))
from common import ENV

def run_repo(label, arguments, log_name, **overrides):
    print('RUN_BEGIN', label, arguments, flush=True)
    log = EVIDENCE / log_name
    with log.open('w', encoding='utf-8') as stream:
        result = subprocess.run([sys.executable, *arguments], cwd=roots[label],
                                env=dict(ENV, **overrides), stdout=stream,
                                stderr=subprocess.STDOUT)
    print(log.read_text(encoding='utf-8'), flush=True)
    result.check_returncode()

def solution_hash(root, python_only=False):
    digest = hashlib.sha256()
    for path in sorted((root / 'solution').rglob('*')):
        if (path.is_file() and '__pycache__' not in path.parts
                and (not python_only or path.suffix == '.py')):
            digest.update(path.relative_to(root / 'solution').as_posix().encode())
            digest.update(path.read_bytes())
    return digest.hexdigest()

provenance = {'runtime_commit': CANDIDATE, 'current_merei_commit': CURRENT_MEREI,
              'versions': versions, 'solution_sha256_before_tests': solution_hash(roots['candidate']),
              'python_sha256': solution_hash(roots['candidate'], python_only=True)}
(EVIDENCE / 'provenance.json').write_text(json.dumps(provenance, indent=2))
print(json.dumps(provenance, indent=2), flush=True)


In [ ]:
run_repo('candidate', ['-m', 'pytest', 'solution', '-q', '-p', 'no:cacheprovider'],
         'solution-tests.log')
run_repo('candidate', ['-m', 'pytest', 'tests', '-q', '-p', 'no:cacheprovider'],
         'soup-regressions.log', SOUP_APPLY_SOLUTION='1')
for label in roots:
    run_repo(label, ['task/prepare_data.py'], label + '-prepare-data.log')
assert solution_hash(roots['candidate']) == provenance['solution_sha256_before_tests']


## Обязательный admission/quality run

Неизменённый harness выполняет base → solution → solution → base и затем
все десять checks, включая Soup regressions. Native Soup здесь нужен для
допуска и loss/outputs comparisons; это отдельная процедура от сравнения
скорости с текущим решением Мерея. Не меняйте packages во время этих запусков.


In [ ]:
ADMISSION = 'runs_reproduce_294_admission'
run_repo('candidate', ['harness/run_task.py', '--repeats', '2', '--out', ADMISSION],
         'official-training.log')
run_repo('candidate', ['harness/check_rules.py', '--result', ADMISSION + '/result.json'],
         'official-checks.log')
run_repo('candidate', ['harness/score.py', ADMISSION + '/result.json'], 'official-score.log')
result_path = roots['candidate'] / ADMISSION / 'result.json'
checks_path = roots['candidate'] / ADMISSION / 'checks.json'
result = json.loads(result_path.read_text())
checks = json.loads(checks_path.read_text())
assert len(checks['checks']) == 10 and all(item['ok'] for item in checks['checks'])
assert checks['result_sha256'] == hashlib.sha256(result_path.read_bytes()).hexdigest()
assert checks['solution_sha256'] == solution_hash(roots['candidate'])
candidate = result['summary']['solution']
assert candidate['gpu_peak_mib'] <= MEMORY_CEILING_MIB, 'Current-Merei memory ceiling exceeded'
shutil.copy2(result_path, EVIDENCE / 'final-result.json')
shutil.copy2(checks_path, EVIDENCE / 'final-checks.json')
print('ADMISSION_COMPLETE', json.dumps({'runtime_commit': CANDIDATE,
      'candidate_summary': candidate, 'checks_passed': 10,
      'memory_ceiling_mib': MEMORY_CEILING_MIB}, indent=2), flush=True)


## Optional: текущий Мерей против runtime 29419f43

Включите `RUN_CURRENT_MEREI_AB=True` в первой code cell до Run All.
Этот этап дополнительно выполняет old → new → new → old без native Soup.
Это performance diagnostic; его result files не заменяют official proof выше.
Не объединяйте seconds с другой сессией и проверяйте whole-command и
steps 3+ отдельно. `--only solution` не предназначен для полного rule check.


In [ ]:
if RUN_CURRENT_MEREI_AB:
    ab_runs = {'current': [], 'candidate': []}
    order = [('current', 'old_1'), ('candidate', 'new_1'),
             ('candidate', 'new_2'), ('current', 'old_2')]
    for label, name in order:
        folder = 'runs_reproduce_ab_' + name
        run_repo(label, ['harness/run_task.py', '--only', 'solution', '--out', folder],
                 'ab-' + name + '.log')
        path = roots[label] / folder / 'result.json'
        report = json.loads(path.read_text())
        assert report['runs']['solution'] and all(r['exit_code'] == 0 for r in report['runs']['solution'])
        ab_runs[label].extend(report['runs']['solution'])
        shutil.copy2(path, EVIDENCE / ('ab-' + name + '-result.json'))

    def summarize(rows):
        return {'wall_runs_s': [r['wall_s'] for r in rows],
                'wall_median_s': statistics.median(r['wall_s'] for r in rows),
                'gpu_peak_mib': max(r['gpu_peak_mib'] for r in rows),
                'train_loop_median_s': statistics.median(r['metrics']['train_loop_s'] for r in rows),
                'timed_steps_3plus_median_s': statistics.median(r['metrics']['timed_loop_s'] for r in rows)}
    summary = {label: summarize(rows) for label, rows in ab_runs.items()}
    assert summary['candidate']['gpu_peak_mib'] <= MEMORY_CEILING_MIB
    assert summary['candidate']['gpu_peak_mib'] <= summary['current']['gpu_peak_mib']
    comparison = {'kind': 'current_merei_ab_diagnostic_not_admission',
                  'reference_commit': CURRENT_MEREI, 'candidate_commit': CANDIDATE,
                  'order': [name for label, name in order], 'summary': summary,
                  'whole_time_reduction_pct': 100 * (1 - summary['candidate']['wall_median_s']
                                                    / summary['current']['wall_median_s'])}
    (EVIDENCE / 'current-merei-ab.json').write_text(json.dumps(comparison, indent=2))
    print(json.dumps(comparison, indent=2), flush=True)
else:
    print('Optional current-Merei A/B disabled; mandatory admission completed above.')


## Evidence и source hashes

`provenance.json`, test logs, final-result.json и final-checks.json лежат в
напечатанной папке evidence. При optional A/B добавляются четыре исходных
diagnostic result files и current-merei-ab.json. Сохраните Kaggle version
с output и скачайте эти файлы. Не публикуйте Kaggle Secrets.

Runtime pinned к 29419f43. Более поздний documentation-only commit может
менять полный solution SHA256: notebook/docs внутри solution тоже входят
в него, хотя production Python остаётся тем же. Этот notebook воспроизводит
bytes выбранного runtime commit; hash более поздней публикации связывается
отдельным check_rules readback, без переноса чужих timing/quality results.


In [ ]:
print('EVIDENCE_DIRECTORY', str(EVIDENCE), flush=True)
print('MEREY_RUNTIME_294_REPRODUCTION_COMPLETE', flush=True)
